导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True


RFI（Radio Frequency Interference）是射电干涉观测里最“现实”的问题之一。它不是来自宇宙，而是来自人类自己：广播、通信、雷达、卫星、航空电子、开关电源，甚至观测站内设备都可能成为干扰源。

本节不再停留在概念层面，而是用三个可以直接重跑的实验来回答：

- RFI 在动态频谱上通常长什么样；
- 一个基础但实用的自动标记流程如何工作；
- 为什么干涉仪在 fringe stopping 之后，对某些地面干扰会有额外抑制。


***


## 7.8 射频干扰（RFI）

RFI 的危险之处不只是“把某些数据点弄脏”。更严重的问题是：

- 它可能是窄带持续的，伪装成谱线或带通结构；
- 也可能是宽带突发的，直接污染整段时间积分；
- 若强度足够高，还会改变统计分布，使常规噪声假设失效；
- 一旦标记过度，又会损失有效积分时间和频率覆盖。

因此，RFI 处理从来不是单纯的“删掉坏点”，而是在**抑制污染**与**保留灵敏度**之间做权衡。


In [ ]:
def running_median_1d(arr, window):
    pad = window // 2
    padded = np.pad(arr, pad, mode="edge")
    view = sliding_window_view(padded, window_shape=window)
    return np.median(view, axis=-1)


def robust_sigma_mad(values):
    values = np.asarray(values)
    return 1.4826 * np.median(np.abs(values - np.median(values)))


def dilate_mask(mask, radius_t=1, radius_f=1):
    out = mask.copy()
    for dt in range(-radius_t, radius_t + 1):
        for df in range(-radius_f, radius_f + 1):
            shifted = np.roll(np.roll(mask, dt, axis=0), df, axis=1)
            if dt > 0:
                shifted[:dt, :] = False
            elif dt < 0:
                shifted[dt:, :] = False
            if df > 0:
                shifted[:, :df] = False
            elif df < 0:
                shifted[:, df:] = False
            out |= shifted
    return out


def safe_nanmedian_by_channel(masked_data, fill_values):
    out = np.empty(masked_data.shape[1], dtype=float)
    for chan in range(masked_data.shape[1]):
        valid = np.isfinite(masked_data[:, chan])
        out[chan] = np.median(masked_data[valid, chan]) if np.any(valid) else fill_values[chan]
    return out


def make_synthetic_dynamic_spectrum(n_time=128, n_freq=256, seed=0):
    rng = np.random.default_rng(seed)
    freq_axis = np.linspace(0.0, 1.0, n_freq)

    bandpass = 1.0 + 0.15 * np.sin(2.0 * np.pi * freq_axis)
    bandpass += 0.08 * np.cos(6.0 * np.pi * freq_axis)
    bandpass *= 1.0 - 0.2 * (freq_axis - 0.5) ** 2

    data = bandpass[None, :] + 0.03 * rng.normal(size=(n_time, n_freq))
    truth_mask = np.zeros_like(data, dtype=bool)

    for channel, amplitude in [(45, 0.8), (120, 1.5), (180, 0.9)]:
        width = 2 if channel != 120 else 3
        sl = slice(channel - width, channel + width + 1)
        data[:, sl] += amplitude
        truth_mask[:, sl] = True

    for t0, amplitude in [(22, 2.0), (86, 1.5)]:
        burst_shape = amplitude * np.exp(
            -0.5 * ((np.arange(n_freq) - n_freq / 2.0) / (0.45 * n_freq)) ** 2
        )
        data[t0 - 1 : t0 + 2, :] += burst_shape
        truth_mask[t0 - 1 : t0 + 2, :] = True

    for t in range(20, 110):
        center = int(40 + 1.5 * (t - 20))
        sl = slice(max(0, center - 3), min(n_freq, center + 4))
        data[t, sl] += 1.2
        truth_mask[t, sl] = True

    for t0, f0 in [(60, 210), (61, 214), (62, 218), (63, 222)]:
        data[t0 - 1 : t0 + 2, f0 - 2 : f0 + 3] += 0.6
        truth_mask[t0 - 1 : t0 + 2, f0 - 2 : f0 + 3] = True

    return data, bandpass, truth_mask


### 7.8.1 动态频谱中的 RFI 形态

在时频平面上，RFI 往往能分成几种典型形态：

- **窄带持续干扰**：固定在少数几个频率通道，长时间存在；
- **宽带突发干扰**：某几个时间片突然把大范围频率都抬高；
- **扫频或啁啾型干扰**：随时间在频率轴上漂移；
- **弱团簇事件**：单看不一定显眼，但会在统计上逐渐累积成问题。

下面构造一个合成动态频谱，把这些形态全部放进同一个示例里。


In [ ]:
dynamic_spectrum, true_bandpass, truth_mask = make_synthetic_dynamic_spectrum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

im0 = axes[0].imshow(dynamic_spectrum, aspect="auto", origin="lower", cmap="viridis")
axes[0].set_title("Synthetic dynamic spectrum with mixed RFI")
axes[0].set_xlabel("frequency channel")
axes[0].set_ylabel("time sample")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(truth_mask.astype(float), aspect="auto", origin="lower", cmap="magma", vmin=0.0, vmax=1.0)
axes[1].set_title("Ground-truth RFI mask used to build the example")
axes[1].set_xlabel("frequency channel")
axes[1].set_ylabel("time sample")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"动态频谱尺寸 = {dynamic_spectrum.shape[0]} x {dynamic_spectrum.shape[1]}")
print(f"真实 RFI 占比 ≈ {truth_mask.mean() * 100:.2f}%")


真实观测里的 RFI 当然通常更复杂，但这个合成例子已经包含了干涉数据处理中最常见的困难之一：**不存在一种单一阈值，可以同时完美处理所有形态的干扰。**


### 7.8.2 一个基础但实用的自动标记流程

实际软件里的 RFI 处理算法很多，例如 AOFlagger 风格的 SumThreshold、统计矩方法、谱峰度（spectral kurtosis）、子空间投影等。这里先实现一个足够基础、但很有代表性的流程：

1. 用时间中值估计每个通道的总体背景；
2. 用频率滑动中值估计平滑带通；
3. 对“通道整体偏高”与“单个时频像素异常高”分别做鲁棒阈值检测；
4. 对检测结果做少量膨胀（dilation），把边缘也一并标记。

这个算法谈不上最先进，但足以把“统计异常检测”和“标记并非纯粹逐点独立”这两个核心思想讲清楚。


In [ ]:
channel_median = np.median(dynamic_spectrum, axis=0)
channel_background = running_median_1d(channel_median, window=21)

channel_residual = channel_median - channel_background
sigma_channel = robust_sigma_mad(channel_residual)
persistent_channel_mask = np.abs(channel_residual) > 8.0 * sigma_channel

residual = dynamic_spectrum - channel_background[None, :]
sigma_pixel = robust_sigma_mad(residual)
z_score = residual / sigma_pixel
pixel_mask = np.abs(z_score) > 7.0

flag_mask = pixel_mask | persistent_channel_mask[None, :]
flag_mask = dilate_mask(flag_mask, radius_t=1, radius_f=1)

flagged_spectrum = np.where(flag_mask, np.nan, dynamic_spectrum)
cleaned_bandpass = safe_nanmedian_by_channel(flagged_spectrum, fill_values=channel_background)

true_positive = np.sum(flag_mask & truth_mask)
false_positive = np.sum(flag_mask & ~truth_mask)
false_negative = np.sum(~flag_mask & truth_mask)
precision = true_positive / (true_positive + false_positive)
recall = true_positive / (true_positive + false_negative)

mae_before = np.mean(np.abs(np.median(dynamic_spectrum, axis=0) - true_bandpass))
mae_after = np.mean(np.abs(cleaned_bandpass - true_bandpass))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

im0 = axes[0, 0].imshow(z_score, aspect="auto", origin="lower", cmap="coolwarm", vmin=-15.0, vmax=15.0)
axes[0, 0].set_title("Robust z-score map")
axes[0, 0].set_xlabel("frequency channel")
axes[0, 0].set_ylabel("time sample")
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

im1 = axes[0, 1].imshow(flag_mask.astype(float), aspect="auto", origin="lower", cmap="gray_r", vmin=0.0, vmax=1.0)
axes[0, 1].set_title("Automatic flag mask")
axes[0, 1].set_xlabel("frequency channel")
axes[0, 1].set_ylabel("time sample")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

axes[1, 0].plot(channel_median, label="median before flagging")
axes[1, 0].plot(cleaned_bandpass, label="median after flagging")
axes[1, 0].plot(true_bandpass, label="true sky bandpass", lw=2.0, color="black", alpha=0.8)
axes[1, 0].set_title("Bandpass recovery")
axes[1, 0].set_xlabel("frequency channel")
axes[1, 0].set_ylabel("amplitude")
axes[1, 0].legend()

axes[1, 1].bar(["true RFI", "flagged"], [truth_mask.mean() * 100.0, flag_mask.mean() * 100.0], color=["tab:orange", "tab:blue"])
axes[1, 1].set_title("Occupancy comparison")
axes[1, 1].set_ylabel("fraction of samples [%]")

fig.tight_layout()
plt.show()
plt.close(fig)

print(f"像素级鲁棒标准差 sigma ≈ {sigma_pixel:.4f}")
print(f"通道级鲁棒标准差 sigma ≈ {sigma_channel:.4f}")
print(f"自动标记占比 ≈ {flag_mask.mean() * 100:.2f}%")
print(f"precision ≈ {precision:.3f}")
print(f"recall    ≈ {recall:.3f}")
print(f"带通中值的平均绝对误差: before = {mae_before:.4f}, after = {mae_after:.4f}")


这组结果很能说明问题：

- 一个“保守型”标记器通常会牺牲一部分精确率，换取更高的召回率；
- 标记过后，带通估计会显著接近真实背景；
- 但代价是数据占空比下降，因此过度标记会直接损失灵敏度。

这也解释了为什么成熟管线不会只给出一个固定阈值，而是会在不同观测、频段、台站环境和科学目标之间做策略调整。


### 7.8.3 干涉仪里的额外抑制：fringe stopping 与积分平均

对相位中心上的天体源，相关器通常会做 fringe stopping，使目标信号在相关后尽量保持“慢变化”或近似静止。相反，某些地面干扰源在做完同样的 fringe stopping 之后，反而会带上明显的残余 fringe rate。

如果一个干扰在相关后可写作

$$
V_{\rm RFI}(t) \propto e^{2\pi i f_{\rm fr} t},
$$

那么在时间积分 $T$ 上的平均响应幅度大约按

$$
|\mathrm{sinc}(f_{\rm fr} T)|
$$

衰减。也就是说，**并不是所有 RFI 都会原封不动地进入最终积分数据**；某些与天体条纹率不匹配的干扰，会在时间平均中部分相消。


In [ ]:
integration_time_s = np.linspace(1.0, 60.0, 240)
fringe_rates_hz = [0.0, 0.02, 0.10, 0.30]

fig, ax = plt.subplots(figsize=(9, 4.8))
for fringe_rate_hz in fringe_rates_hz:
    attenuation = np.abs(np.sinc(fringe_rate_hz * integration_time_s))
    label = "phase-centre sky source" if fringe_rate_hz == 0.0 else f"RFI fringe rate = {fringe_rate_hz:.2f} Hz"
    ax.plot(integration_time_s, attenuation, lw=2.0, label=label)

ax.set_xlabel("integration time [s]")
ax.set_ylabel("post-average amplitude response")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Fringe stopping suppresses non-matching fringe rates")
ax.legend()

fig.tight_layout()
plt.show()
plt.close(fig)

print("10 秒积分时的响应保留比例：")
for fringe_rate_hz in fringe_rates_hz:
    attenuation_10s = np.abs(np.sinc(fringe_rate_hz * 10.0))
    if fringe_rate_hz == 0.0:
        print(f"  相位中心天体源 -> {attenuation_10s:.3f}")
    else:
        print(f"  fringe rate = {fringe_rate_hz:.2f} Hz -> {attenuation_10s:.3f}")


这里要特别注意两点：

- 这不是“RFI 会自动消失”的万能机制，因为很多干扰仍然可能与阵列几何部分相干；
- 但它确实说明，干涉仪和单天线在 RFI 表现上并不完全一样，相关与积分本身就会引入额外的选择性。

因此，真正的 RFI 处理通常是多层次的：观测前的频谱管理、观测中的硬件抑制、相关前后的统计标记、以及后续成像时对 flag 的正确传播，都缺一不可。
